In [ ]:

import matplotlib.pyplot as plt
import nibabel as nib
import pandas as pd
import numpy as np
import scipy.io 
import mne
import os
import sys

tfr_file_path = "E:/workspace/study2_escape_task_eeg/tfr_data/" # 分析哪个roi就输入哪个文件夹的数据
tfr_type = "all_trial_data_zlogratio/" 
tfr_path = tfr_file_path + tfr_type

behavior_path = "E:/workspace/study2_escape_task_eeg/behavior/" # 分析哪个roi就输入哪个文件夹的数据


subjects = [4, 5, 6, 8, 9, 11, 12, 14, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38,39,40,41,42,43,45, 46, 47,49,50,51, 52] 

42

### 要先确保eeg数据epoch的顺序和behavior的顺序是按序列一一对应的

In [7]:
channel_location_dict = {"Fp1":	[-26.25, 83, -18.25],
"Fp2": [26.25, 83, -18.25],
"F3": [-48.75, 57.5, 36.5],
"F4": [48.75, 57.5, 36.5],
"C3": [-63.75, -13, 65.75],
"C4": [63.75, -13, 65.75],
"P3": [-48, -86.5, 53],
"P4": [48, -86.5, 53],
"O1": [-27, -118, 4.25],
"O2": [27, -118, 4.25],
"F7": [-69, 44, -18.25],
"F8": [69, 44, -18.25],
"T7": [-84, -18.25, -13],
"T8": [84, -18.25, -13],
"P7": [-69, -80.5, -4.75],
"P8": [69, -80.5, -4.75],
"Fz": [0, 63.5, 59],
"Cz": [0, -10, 99.5],
"Pz": [0, -88, 77],
"FC1": [-32.25, 28.25, 77.75],
"FC2": [32.25, 28.25, 77.75],
"CP1": [-35.25, -52, 90.5],
"CP2": [35.25, -52, 90.5],
"FC5": [-75.75, 20, 21.5],
"FC6": [75.75, 20, 21.5],
"CP5": [-78, -51.25, 31.25],
"CP6": [78, -51.25, 31.25],
"FT9": [-72.75, 8.75, -55.75],
"FT10": [72.75, 8.75, -55.75],
"TP9": [-71.25, -49, -49.75],
"TP10": [71.25, -49, -49.75],
"F1": [-25.5, 62, 53.75],
"F2": [25.5, 62, 53.75],
"C1": [-35.25, -11.5, 89.75],
"C2": [35.25, -11.5, 89.75],
"P1": [-27, -88, 70.25],
"P2": [27, -88, 70.25],
"AF3": [-32.25, 80.75, 11.75],
"AF4": [32.25, 80.75, 11.75],
"FC3": [-59.25, 25.25, 54.5],
"FC4": [59.25, 25.25, 54.5],
"CP3": [-63, -52, 66.5],
"CP4": [63, -52, 66.5],
"PO3": [-31.5, -109.75, 32.75],
"PO4": [31.5, -109.75, 32.75],
"F5": [-63, 51.5, 11.75],
"F6": [63, 51.5, 11.75],
"C5": [-81.75, -14.5, 29],
"C6": [81.75, -14.5, 29],
"P5": [-63.75, -83.5, 26.75],
"P6": [63.75, -83.5, 26.75],
"AF7": [-50.25, 68, -19],
"AF8": [50.25, 68, -19],
"FT7": [-79.5, 15.5, -16],
"FT8": [79.5, 15.5, -16],
"TP7": [-80.25, -49.75, -8.5],
"TP8": [80.25, -49.75, -8.5],
"PO7": [-50.25, -103, 0.5],
"PO8": [50.25, -103, 0.5],
"Fpz": [0, 83, -18.25],
"CPz": [0, -53.5, 98],
"POz": [0, -109, 44],
"Oz": [0, -122.5, 8]}
# ['Fp1', 'Fp2', 'Fz', 'Cz', 'Pz', 'Fpz', 'CPz', 'POz', 'Oz'].
mymontage = mne.channels.make_dig_montage(ch_pos=channel_location_dict)#, nasion=[0, -10, 99.5], lpa=[-71.25, -49, -49.75], 
#                                        rpa=[71.25, -49, -49.75])

In [8]:
from joblib import Parallel, delayed
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

def compute_cluster_p_value(cluster_stat, cluster_stats_H0):

    return sum(ms > (cluster_stat) for ms in cluster_stats_H0) / len(cluster_stats_H0)

def find_1d_cluster(t_maps, p_maps, p_thresh=0.05, cluster_min_size=2, tail=0):
    # 输出的数据为[(起始位置, 结束位置+1), (起始位置, 结束位置+1)]
    if tail == 1:
        mask = ((t_maps > 0) & (p_maps < p_thresh)).astype(int)
    elif tail == -1:
        mask = ((t_maps < 0) & (p_maps < p_thresh)).astype(int)
    elif tail == 0:
        mask = (p_maps < p_thresh).astype(int)
    # 找连续的1
    start = -1  # 起始位置
    length = 0  # 连续1的长度
    results = []  # 存储结果

    for i in range(len(mask)):
        if mask[i] == 1:
            if start == -1:
                start = i
            length += 1
        else:
            if length >= cluster_min_size:
                results.append((start, start + length))
            start = -1
            length = 0

    # 处理连续1在数组末尾的情况
    if length >= cluster_min_size:
        results.append((start, start + length))

    return results


def compute_cluster_stats(stat_map, clusters):
    return [abs(stat_map[cluster[0]: cluster[1]].sum()) for cluster in clusters]

def one_d_glm(regress_data, regressors_design_dict, model_formula):
    def fit_model(iter_time_point):
        data = regressors_design_dict.copy()
        data['EEG'] = regress_data[:, iter_time_point]
        model = smf.ols(formula=model_formula, data=data)
        result = model.fit()
        conf_int = result.conf_int().values  # Get confidence intervals
        return result.params.values, result.tvalues.values, result.pvalues.values, conf_int, result.bse.values

    num_cores = 4
    results = Parallel(n_jobs=num_cores)(delayed(fit_model)(iter_time_point) for iter_time_point in range(regress_data.shape[1]))

    paramap, tmap, pmap, conf_int_map, bse_map = zip(*results)
    return np.array(paramap), np.array(tmap), np.array(pmap), np.array(conf_int_map), np.array(bse_map)



In [9]:
def one_d_glm_sts(regress_data, regressors_design_dict, model_formula):
    def fit_model(iter_time_point):
        data = regressors_design_dict.copy()
        # 初始化 StandardScaler
        scaler = StandardScaler()
        # 对数据进行标准化
        scaled_data = scaler.fit_transform(data)
        # 将标准化后的数据转换为 DataFrame
        scaled_df = pd.DataFrame(scaled_data, columns=data.keys())
        scaled_df['EEG'] = regress_data[:, iter_time_point]
        model = smf.ols(formula=model_formula, data=scaled_df)
        result = model.fit()
        conf_int = result.conf_int().values  # Get confidence intervals
        return result.params.values, result.tvalues.values, result.pvalues.values, conf_int, result.bse.values

    num_cores = 6
    results = Parallel(n_jobs=num_cores)(delayed(fit_model)(iter_time_point) for iter_time_point in range(regress_data.shape[1]))

    paramap, tmap, pmap, conf_int_map, bse_map = zip(*results)
    return np.array(paramap), np.array(tmap), np.array(pmap), np.array(conf_int_map), np.array(bse_map)


In [ ]:
#个体水平
from mne.time_frequency import tfr_morlet
from tqdm import *
time_span = 100
ch_names = []
decision_time_range = [-1.0, 5] # decision前后  0.9
time_range = [-0.5, 1.0]  # baseline前后 0.3s
# subjects=[47]
all_sub_paramap = []
all_sub_tmap = []
all_sub_pmap = []
imminent_trial_num, moderate_trial_num = [], []

# 用于存储所有被试的 VIF 结果
final_results = {}

for i in range(len(subjects)):

    # 导入数据 删除眼电 设置电极位置
    EEG_epochs = mne.read_epochs(tfr_path  + str(subjects[i]) + 'tfr.fif', verbose=False)

    # 导入行为数据
    hebavior_trial = pd.read_excel(behavior_path + str(subjects[i]) + '/subject_v5.xlsx')
    # 补充编号序列用来后续提出epoch
    hebavior_trial['index'] = range(len(hebavior_trial))

    # 定义需要保留的列
    columns_to_keep = [
        's_safe_first_distance_stage_3',
        's_p_first_distance_stage_3',
        'rt_stage34',
        'mean_dv_stage2',
        'mean_dv_stage3',
        'mean_dv_stage4',
        'stationary_ratio_stage_3',
        'event_num',
        'index',
        'get_caught',
        'get_safe',
        'too_fast',
        'trial_time',
        'firsttime_stage_2',
        'firsttime_stage_3',
        'firsttime_stage_4',
        'firsttime_stage_5',
        'remain_square_num_stage3_first'
    ]

    # 只保留指定的列
    hebavior_trial = hebavior_trial[columns_to_keep]
    # print((hebavior_trial))

    # 检查指定列中是否存在 NaN 或空集 []，并删除这些行
    columns_to_check = [
    's_safe_first_distance_stage_3',
    'rt_stage34',
    'mean_dv_stage3',
    'stationary_ratio_stage_3',
    'firsttime_stage_3',
    's_p_first_distance_stage_3',
    'remain_square_num_stage3_first'
    ]
    
    mask = (
        hebavior_trial[columns_to_check].isna() |  # 检查 NaN
        hebavior_trial[columns_to_check].applymap(lambda x: x == [])  # 检查空集 []
            ).any(axis=1)  # 如果任意一列满足条件，则标记为 True

    # 删除满足条件的行
    hebavior_trial = hebavior_trial[~mask]

    safe_direct_event_num = np.squeeze(np.where((hebavior_trial['event_num']==22)))
    safe_divert_event_num = np.squeeze(np.where(( hebavior_trial['event_num']==32) | (hebavior_trial['event_num']==31)) ) 

    direct_behavior = hebavior_trial.iloc[safe_direct_event_num]
    divert_behavior = hebavior_trial.iloc[safe_divert_event_num]

    ################################ stage 1
    direct_epoch = EEG_epochs.copy()[np.array(direct_behavior['index'])]
    divert_epoch = EEG_epochs.copy()[np.array(divert_behavior['index'])]
    direct_epoch.crop(time_range[0], time_range[1])
    divert_epoch.crop(time_range[0], time_range[1])
    
    ################################ stage 3
    # 高威胁攻击
    discarded_trial_1 = []
    empty_or_not = 0
    for iter_index in np.array(direct_behavior['index']):

        # 结合每个trial的数据确定每个trial要截取的时间窗
        iter_trialtime = time_range + np.array(round(float(direct_behavior['firsttime_stage_3'][iter_index]),2))
        if iter_trialtime[1] < 16 - 0.5: # 时间太长的就不用了
            # 利用mne的函数截取数据
            iter_safe_imminent_data = EEG_epochs.copy()[iter_index].crop(iter_trialtime[0], iter_trialtime[1]-0.010).get_data()

            # 存储数据
            if empty_or_not == 0: 
                iter_subject_safe_imminent_data = iter_safe_imminent_data
                empty_or_not = 1
            else: 
                iter_subject_safe_imminent_data = np.vstack((iter_subject_safe_imminent_data, iter_safe_imminent_data))
        else:
            discarded_trial_1.append(iter_index)

    # 低威胁攻击
    discarded_trial_2 = []
    empty_or_not = 0
    for iter_index in np.array(divert_behavior['index']):

        # 结合每个trial的数据确定每个trial要截取的时间窗
        iter_trialtime = time_range + np.array(round(float(divert_behavior['firsttime_stage_3'][iter_index]),2))
        if iter_trialtime[1] < 16-0.5: # 时间太长的就不用了
            # 利用mne的函数截取数据
            iter_safe_moderate_data = EEG_epochs.copy()[iter_index].crop(iter_trialtime[0], iter_trialtime[1]-0.010).get_data()

            # 存储数据
            if empty_or_not == 0: 
                iter_subject_safe_moderate_data = iter_safe_moderate_data
                empty_or_not = 1
            else: 
                iter_subject_safe_moderate_data = np.vstack((iter_subject_safe_moderate_data, iter_safe_moderate_data))
        else:
            discarded_trial_2.append(iter_index)


    # 用mne数据格式存储
    info = mne.create_info(ch_names = EEG_epochs.ch_names, ch_types = 'eeg', sfreq = EEG_epochs.info['sfreq'])

    # 矫正前平均
    iter_sub_imminent_epoch = mne.EpochsArray(data = iter_subject_safe_imminent_data, info = info, tmin=time_range[0])
    iter_sub_moderate_epoch = mne.EpochsArray(data = iter_subject_safe_moderate_data, info = info, tmin=time_range[0])
    # iter_sub_imminent_epoch.apply_baseline(baseline = (-0.5,-0.1))
    # iter_sub_moderate_epoch.apply_baseline(baseline = (-0.5,-0.1)) 
    if len(discarded_trial_1) > 0:
        direct_behavior.drop(index=discarded_trial_1, inplace=True)
    if len(discarded_trial_2) > 0:
        divert_behavior.drop(index=discarded_trial_2, inplace=True)
    
    imminent_trial_num.append(len(direct_behavior))
    moderate_trial_num.append(len(divert_behavior))
    ## 开始回归

    from tqdm import tqdm

    model_formula = 'EEG ~ mean_dv_stage3 + rt_stage34 + s_safe_first_distance_stage_3 + stationary_ratio_stage_3 + remain_square_num_stage3_first'
    # model_formula = 'EEG ~ low_p_psafe_distance'


    #################### direct
    columns_to_keep = [
        'mean_dv_stage3',
        'rt_stage34',
        's_safe_first_distance_stage_3',
        'stationary_ratio_stage_3',
        'remain_square_num_stage3_first'
    ]
    # columns_to_keep = [
    #     'mean_dv_stage3',
    #     'rt_stage34',
    #     's_p_first_distance_stage_3',
    #     'stationary_ratio_stage_3',
    # ]

    # 选择回归数据，并且只保留指定的列
    regressors_dict = direct_behavior[columns_to_keep]
    EEG_epoch_data = iter_sub_imminent_epoch
    # regressors_dict = divert_behavior[columns_to_keep]
    # EEG_epoch_data = iter_sub_moderate_epoch

    ######## 计算VIF
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from statsmodels.tools.tools import add_constant
    # 添加常数项
    X = add_constant(regressors_dict)
    
    # 确保数据是浮点类型
    X = X.astype(float)
    
    # 计算 VIF
    vif_results = {
        column: variance_inflation_factor(X.values, i)
        for i, column in enumerate(X.columns) if column != 'const'
    }
    

    final_results[i] = vif_results
    final_vif_df = pd.DataFrame(final_results).T 


    # 把移动向量反过来
    regressors_dict['mean_dv_stage3'] = -regressors_dict['mean_dv_stage3']
    iter_sub_paramap, iter_sub_tmap, iter_sub_pmap = [], [], []
    for iter_channel in trange(len(EEG_epoch_data.ch_names)):
        regress_data = EEG_epoch_data.get_data()[:,iter_channel,:]
        # 普通回归
        paramap, tmap, pmap, _, _ = one_d_glm_sts(regress_data, regressors_design_dict=regressors_dict, model_formula=model_formula)

        iter_sub_paramap.append(paramap)
        iter_sub_tmap.append(tmap)
        iter_sub_pmap.append(pmap)

    all_sub_paramap.append(iter_sub_paramap)
    all_sub_tmap.append(iter_sub_tmap)
    all_sub_pmap.append(iter_sub_pmap)
    # 第一行是截距

#### 保存vif结果
# 将结果保存到 CSV 文件
final_vif_df.to_csv('E:/workspace/study2_escape_task_eeg/VIF_result/vif_direct_results.csv', index=False)

# print("VIF results saved to 'vif_results.csv'")

In [ ]:
# 存储lasso回归数据
all_sub_paramap = np.array(all_sub_paramap)
all_sub_tmap = np.array(all_sub_tmap)
all_sub_pmap = np.array(all_sub_pmap)
# 用mne的数据格式进行存储
info = mne.create_info(ch_names = EEG_epochs.ch_names, ch_types = 'eeg', sfreq = EEG_epochs.info['sfreq'])
# nobaseline_ols_direct_ynozscore 无岭回归   nobaseline_elastic_ols_direct_ynozscore 有岭回归
result_path = tfr_path + 'nobaseline_ols_direct_ynozscore/'   # baseline_elastic_ols_direct   nobaseline_elastic_ols_direct_ynozscore
# 检查路径是否存在
if not os.path.exists(result_path):
    # 如果路径不存在，则创建文件夹
    os.makedirs(result_path)
# mean_dv_stage3
mean_dv_stage3_paramap = mne.EpochsArray(data = all_sub_paramap[:,:,:,1], info = info, tmin=time_range[0])
# mean_dv_stage3_tmap = mne.EpochsArray(data = all_sub_tmap[:,:,:,1], info = info, tmin=time_range[0])
# mean_dv_stage3_pmap = mne.EpochsArray(data = all_sub_pmap[:,:,:,1], info = info, tmin=time_range[0])
mean_dv_stage3_paramap.save(result_path +  'mean_dv_stage3_paramap.fif', overwrite=True)
# mean_dv_stage3_tmap.save(result_path +  'mean_dv_stage3_tmap.fif', overwrite=True)
# mean_dv_stage3_pmap.save(result_path +  'mean_dv_stage3_pmap.fif', overwrite=True)

# rt_stage34
rt_stage34_paramap = mne.EpochsArray(data = all_sub_paramap[:,:,:,2], info = info, tmin=time_range[0])
# rt_stage34_tmap = mne.EpochsArray(data = all_sub_tmap[:,:,:,2], info = info, tmin=time_range[0])
# rt_stage34_pmap = mne.EpochsArray(data = all_sub_pmap[:,:,:,2], info = info, tmin=time_range[0])
rt_stage34_paramap.save(result_path +  'rt_stage34_paramap.fif', overwrite=True)
# rt_stage34_tmap.save(result_path +  'rt_stage34_tmap.fif', overwrite=True)
# rt_stage34_pmap.save(result_path +  'rt_stage34_pmap.fif', overwrite=True)

# s_safe_first_distance_stage_3
s_safe_first_distance_stage_3_paramap = mne.EpochsArray(data = all_sub_paramap[:,:,:,3], info = info, tmin=time_range[0])
# s_safe_first_distance_stage_3_tmap = mne.EpochsArray(data = all_sub_tmap[:,:,:,3], info = info, tmin=time_range[0])
# s_safe_first_distance_stage_3_pmap = mne.EpochsArray(data = all_sub_pmap[:,:,:,3], info = info, tmin=time_range[0])
s_safe_first_distance_stage_3_paramap.save(result_path +  's_safe_first_distance_stage_3_paramap.fif', overwrite=True)
# s_safe_first_distance_stage_3_tmap.save(result_path +  's_safe_first_distance_stage_3_tmap.fif', overwrite=True)
# s_safe_first_distance_stage_3_pmap.save(result_path +  's_safe_first_distance_stage_3_pmap.fif', overwrite=True)

# s_safe_first_distance_stage_3
stationary_ratio_stage_3_paramap = mne.EpochsArray(data = all_sub_paramap[:,:,:,4], info = info, tmin=time_range[0])
# stationary_ratio_stage_3_tmap = mne.EpochsArray(data = all_sub_tmap[:,:,:,4], info = info, tmin=time_range[0])
# stationary_ratio_stage_3_pmap = mne.EpochsArray(data = all_sub_pmap[:,:,:,4], info = info, tmin=time_range[0])
stationary_ratio_stage_3_paramap.save(result_path +  'stationary_ratio_stage_3_paramap.fif', overwrite=True)
# stationary_ratio_stage_3_tmap.save(result_path +  'stationary_ratio_stage_3_tmap.fif', overwrite=True)
# stationary_ratio_stage_3_pmap.save(result_path +  'stationary_ratio_stage_3_pmap.fif', overwrite=True)

# remain_square_num_stage3_first
remain_square_num_stage3_first_paramap = mne.EpochsArray(data = all_sub_paramap[:,:,:,5], info = info, tmin=time_range[0])
# mean_dv_stage3_tmap = mne.EpochsArray(data = all_sub_tmap[:,:,:,1], info = info, tmin=time_range[0])
# mean_dv_stage3_pmap = mne.EpochsArray(data = all_sub_pmap[:,:,:,1], info = info, tmin=time_range[0])
remain_square_num_stage3_first_paramap.save(result_path +  'remain_square_num_stage3_first_paramap.fif', overwrite=True)
# mean_dv_stage3_tmap.save(result_path +  'mean_dv_stage3_tmap.fif', overwrite=True)
# mean_dv_stage3_pmap.save(result_path +  'mean_dv_stage3_pmap.fif', overwrite=True)

In [ ]:
# 提取数据

# 定义路径
result_path = 'E:/workspace/study2_escape_task_eeg/tfr_data/all_trial_data_zlogratio_theta_mnebaseline/nobaseline_ols_direct_ynozscore/'

# 定义文件名
file_names = {
    "mean_dv_stage3_paramap": result_path + 'mean_dv_stage3_paramap.fif',
    "rt_stage34_paramap": result_path + 'rt_stage34_paramap.fif',
    "s_safe_first_distance_stage_3_paramap": result_path + 's_safe_first_distance_stage_3_paramap.fif',
    "stationary_ratio_stage_3_paramap": result_path + 'stationary_ratio_stage_3_paramap.fif',
}

# 创建一个字典来存储加载的数据
loaded_data = {}

# 循环加载文件
for key, file_path in file_names.items():
    print(f"Loading {key} from {file_path}")
    loaded_data[key] = mne.read_epochs(file_path, verbose=False)

# 提取数据数组
mean_dv_stage3_paramap = loaded_data["mean_dv_stage3_paramap"] # 提取原始数据为 numpy 数组

rt_stage34_paramap = loaded_data["rt_stage34_paramap"]

s_safe_first_distance_stage_3_paramap = loaded_data["s_safe_first_distance_stage_3_paramap"]

stationary_ratio_stage_3_paramap = loaded_data["stationary_ratio_stage_3_paramap"]

remain_square_num_stage3_first_paramap = loaded_data["remain_square_num_stage3_first_paramap"]



In [ ]:
# spatio_temporal_cluster_test
from mne.stats import *
import mne
from mne.channels import find_ch_adjacency
from mne.datasets import sample
from mne.stats import combine_adjacency, spatio_temporal_cluster_test
from mne.viz import plot_compare_evokeds
n_permutations = 10000

# anova
# 两者之间的显著性
# T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_test([all_smooth_hgb_slow_crop.data, all_smooth_hgb_fast_crop.data],
#                                                                 out_type='mask', n_permutations=n_permutations, n_jobs=6,tail=0
#                                                                 ,verbose=None, t_power=0) #, stat_fun=scipy.stats.ttest_ind())
data1 = s_safe_first_distance_stage_3_paramap


# all_smooth_hgb_fast = all_smooth_hgb_fast.crop(-0.1, 1.1)
# all_smooth_hgb_slow = all_smooth_hgb_slow.crop(-0.1, 1.1)

X = [
    data1.get_data()
]
data1.set_montage(mymontage)
adjacency, ch_names = find_ch_adjacency(data1.info, ch_type="eeg")


def stat_fun_ttest_ind(X,Y):
    return scipy.stats.ttest_rel(X,Y).statistic
from scipy.stats import t
degrees_of_freedom = data1.get_data().shape[0] - 1
t_value_threshold = t.ppf(1 - 0.025, degrees_of_freedom)
# print(t_value_threshold)
data1.crop(0,1)
t_obs, clusters, cluster_pv, h0 = spatio_temporal_cluster_1samp_test(
    data1.get_data().transpose(0, 2, 1),
    n_permutations=n_permutations,
    tail=0,
    n_jobs=6,
    buffer_size=None,
    adjacency=adjacency,
    t_power=1,
    out_type = 'mask',
    # threshold=t_value_threshold,
)

In [ ]:
from mne.channels import find_ch_adjacency, make_1020_channel_selections


all_mask = np.zeros((t_obs.shape), dtype=bool)
for i in range(len(cluster_pv)):
    if cluster_pv[i] < 0.05:
        all_mask = all_mask | clusters[i]
# all_mask = 
# all_mask = clusters[1]
# We need an evoked object to plot the image to be masked
evoked = data1.average()
evoked.set_montage(mymontage)

time_unit = dict(time_unit="s")
figsize=(32,16) 
evoked.plot_joint(
    title="imminent vs. moderate", ts_args=time_unit, topomap_args=time_unit, times=[ 0, 0.2, 0.3, 0.4, 0.6, 0.7]
)  # show difference wave

# Create ROIs by checking channel labels

selections = make_1020_channel_selections(evoked.info, midline="12z")


figsize=(20,12) 
dpi = 100
fig, ax = plt.subplots(figsize=(20, 10))
# Visualize the results
evoked.plot_image(
    axes=ax,
    colorbar=False,
    show=True,
    mask=all_mask.T,
    # show_names="all",
    # mask_style="contour",
    titles=None,
    **time_unit, 
)
# for image in ax.images:  # 遍历轴上的所有图像
#     image.set_clim(vmin=-100000, vmax=100000)  # 设置颜色范围


# evoked = res["x1"].beta
# fig, axes = plt.subplots(figsize=(16,5))
# plt.rcParams['figure.figsize'] = (20,12) # 设置figure_size尺寸
# fig = evoked.plot_image(axes=axes, mask=all_mask, time_unit='s')#, xlim=[-0.05, 0.2])
# plt.set_size_inches(10)


plt.xlabel('Time (s)', fontsize=16)  # 调整标签字体大小
# plt.colorbar(axes["Left"].images[-1], ax=list(axes.values()), shrink=0.3, label="µV")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.close('all')
# times = [0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]  # 需要绘制的时间点列表
times = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, ]  # 需要绘制的时间点列表

# all_mask = np.full(res["x1"].p_val.data.shape, False)
# all_mask[clusters[0][1], clusters[0][0]] = True
# all_mask[clusters[1][1], clusters[1][0]] = True

# time_mask = np.squeeze(all_mask.T[:,timepoint])

fig = evoked.plot_topomap(
    mask = all_mask.T, 
    outlines='head',
    times=times, 
    size=6, 
    show_names=False, 
    cmap='RdBu_r',
    vmin=-200000,
    vmax=200000,
    scalings=None, 
    sensors='ko',
    # sphere=0.13,
    # extrapolate='head',
    colorbar=False,
    mask_params=dict(marker='o',markersize=5, markerfacecolor='black')
    )# cmap GnBu